In [3]:
import asyncio
from autogen_core.models import UserMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
model_client = OpenAIChatCompletionClient(
    model="gemini-2.5-flash",
    api_key=api_key,
)


In [4]:
from autogen_agentchat.agents import AssistantAgent

dsa_solver = AssistantAgent(
    name = 'Complex_DSA_Solver',
    model_client=model_client,
    description='A DSA solver',
    system_message="You give code in python to solve complex DSA problems. Give under 100 words."
)

code_reviewer = AssistantAgent(
    name = 'CODE_REVIEWER',
    model_client=model_client,
    description='A Code Reviewer',
    system_message="You review the code given by the complex_dsa_solver and make sure it is optimized.Give under 10 words. If you feel that the code is fine, please say 'TERMINATE'"
)

code_editor = AssistantAgent(
    name = 'CODE_EDITOR',
    model_client=model_client,
    description='A Code editor',
    system_message="You make the code easy to understand and add comments wherever required.Give under 10 words"
)

In [7]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage

team = RoundRobinGroupChat(
    participants=[dsa_solver,code_reviewer,code_editor],
    max_turns=3
)

In [8]:
async def run_team():
    task = TextMessage(content="write a simple code in python to add 2 numbers",source="user")
    result = await team.run(task=task)
    print(result)

await run_team()

messages=[TextMessage(id='fe9b085e-ffec-40d5-bb0a-fc7e0a5c5f4c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 6, 12, 56, 25, 464282, tzinfo=datetime.timezone.utc), content='write a simple code in python to add 2 numbers', type='TextMessage'), TextMessage(id='aac0f399-529d-4e08-88cb-811ff60e4373', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=32, completion_tokens=183), metadata={}, created_at=datetime.datetime(2025, 10, 6, 12, 56, 27, 992929, tzinfo=datetime.timezone.utc), content='```python\ndef add_two_numbers(num1, num2):\n  """Adds two numbers and returns the sum."""\n  return num1 + num2\n\n# Example usage:\nnumber1 = 5\nnumber2 = 10\nsum_result = add_two_numbers(number1, number2)\nprint(f"The sum of {number1} and {number2} is: {sum_result}")\n\n# You can also get input from the user:\n# try:\n#     val1 = float(input("Enter first number: "))\n#     val2 = float(input("Enter second number: "))\n#     print(f"The sum i

In [10]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination


my_termination = TextMentionTermination(text='TERMINATE') 

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination,
    max_turns=6
)


async def run_team():
    task = TextMessage(content='write a code to find median of two sorted arrays',source='user')

    result = await team.run(task=task)

    for each_agent_message in result.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")


    # print(result)

await run_team()

user : write a code to find median of two sorted arrays
Complex_DSA_Solver : This code finds the median of two sorted arrays, `nums1` and `nums2`, in `O(log(min(m,n)))` time. It employs a binary search strategy to find the optimal partition point in the shorter array, ensuring all elements in the left half of the combined array are less than or equal to all elements in the right half.

```python
def findMedianSortedArrays(nums1, nums2):
    if len(nums1) > len(nums2):
        nums1, nums2 = nums2, nums1  # Ensure nums1 is the shorter array

    m, n = len(nums1), len(nums2)
    low, high = 0, m

    while low <= high:
        partitionX = (low + high) // 2
        partitionY = (m + n + 1) // 2 - partitionX

        maxLeftX = float('-inf') if partitionX == 0 else nums1[partitionX - 1]
        minRightX = float('inf') if partitionX == m else nums1[partitionX]
        maxLeftY = float('-inf') if partitionY == 0 else nums2[partitionY - 1]
        minRightY = float('inf') if partitionY == 

In [9]:

from autogen_agentchat.base import TaskResult
from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination


my_termination = TextMentionTermination(text='TERMINATE') 

team_2 = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination,
    max_turns=6
)


async for message in team_2.run_stream(task="Write a simple Hello world code ?"):  # type: ignore

    print(type(message))
    if isinstance(message, TaskResult):
        print("Stop Reason:", message.stop_reason)
    else:
        print(message.source,message)

<class 'autogen_agentchat.messages.TextMessage'>
user id='31ee8069-1093-4467-a5f9-6c82d7eeffe5' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 10, 6, 13, 0, 46, 528847, tzinfo=datetime.timezone.utc) content='Write a simple Hello world code ?' type='TextMessage'
<class 'autogen_agentchat.messages.TextMessage'>
Complex_DSA_Solver id='8c1e9c0d-7471-4204-b231-aab6b41f2aa0' source='Complex_DSA_Solver' models_usage=RequestUsage(prompt_tokens=224, completion_tokens=11) metadata={} created_at=datetime.datetime(2025, 10, 6, 13, 0, 47, 930709, tzinfo=datetime.timezone.utc) content='```python\nprint("Hello, World!")\n```' type='TextMessage'
<class 'autogen_agentchat.messages.TextMessage'>
CODE_REVIEWER id='93b048e0-0888-452b-af64-0ba7ba8364ec' source='CODE_REVIEWER' models_usage=RequestUsage(prompt_tokens=262, completion_tokens=2) metadata={} created_at=datetime.datetime(2025, 10, 6, 13, 0, 48, 867670, tzinfo=datetime.timezone.utc) content='TERMINATE' type='TextMes

In [13]:
team.reset()

<coroutine object BaseGroupChat.reset at 0x000001A2057C3790>

In [14]:

from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number you got from previous run. Give result as output."
)
 
add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number from previous run. Give result as output."
)

my_increment_team = RoundRobinGroupChat(participants=[add_1_agent_first,add_1_agent_second,add_1_agent_third],max_turns=2)

In [ ]:
from autogen_agentchat.ui import Console

await Console(my_increment_team.run_stream())

In [ ]:
await Console(my_increment_team.run_stream())

In [16]:
from autogen_core import CancellationToken


# Create a cancellation token.
cancellation_token = CancellationToken()

# Use another coroutine to run the team.
run = asyncio.create_task(
    team.run(
        task="Translate the poem to Spanish.",
        cancellation_token=cancellation_token,
    )
)

# Cancel the run.
cancellation_token.cancel()

try:
    result = await run  # This will raise a CancelledError.
except asyncio.CancelledError as e:
    print(e)
    print("Task was cancelled.")


Task was cancelled.
